In [1]:
# Gut check: does adding the rest of Southeast Asia change the chokepoint ranking?
import pandas as pd
import networkx as nx

df = pd.read_csv("data/ctdc_global_synthetic_data_v2026.csv", low_memory=False)
sex_df = df[df["isSexualExploit"] == 1]

print("All records:", len(df))
print("Sexual exploitation:", len(sex_df))

All records: 268515
Sexual exploitation: 100390


## What this notebook tests

**The boundary in the main analysis.** The country set was seven Southeast Asian ORIGIN countries (PHL, THA, IDN, VNM, KHM, MMR, LAO), chosen at the proposal stage on Renshaw (2016), who maps regional flows from those countries toward Malaysia and Singapore. Malaysia, Singapore, Brunei and Timor-Leste are also Southeast Asian but were classified as the destination side and left out of the set.

**Why that matters.** The filter keeps a record only if at least one endpoint is in the country set. A Malaysia-to-USA

In [2]:
sea7 = ["PHL", "THA", "IDN", "VNM", "KHM", "MMR", "LAO"]
sea11 = sea7 + ["MYS", "SGP", "BRN", "TLS"]

for name, codes in [("SEA-7 (current)", sea7), ("SEA-11 (gut check)", sea11)]:
    either = sex_df[sex_df["citizenship"].isin(codes) | sex_df["CountryOfExploitation"].isin(codes)]
    complete = either.dropna(subset=["citizenship", "CountryOfExploitation"])
    cross = complete[complete["citizenship"] != complete["CountryOfExploitation"]]
    print(f"{name}: {len(either)} in scope | {len(complete)} both endpoints | {len(cross)} cross-border")

SEA-7 (current): 4315 in scope | 2267 both endpoints | 1271 cross-border
SEA-11 (gut check): 4474 in scope | 2300 both endpoints | 1304 cross-border


In [3]:
either = sex_df[sex_df["citizenship"].isin(sea11) | sex_df["CountryOfExploitation"].isin(sea11)]
complete = either.dropna(subset=["citizenship", "CountryOfExploitation"])

edges = complete.groupby(["citizenship", "CountryOfExploitation"]).size().reset_index(name="weight")
edges = edges[edges["citizenship"] != edges["CountryOfExploitation"]]

G11 = nx.from_pandas_edgelist(
    edges, source="citizenship", target="CountryOfExploitation",
    edge_attr="weight", create_using=nx.DiGraph()
)

print("SEA-11 network:", G11.number_of_nodes(), "nodes,", G11.number_of_edges(), "edges")
print("(main analysis was 71 nodes, 158 edges)")

for c in ["MYS", "SGP", "BRN", "TLS"]:
    if c in G11:
        print(f"{c}: inbound {int(G11.in_degree(c, weight='weight'))}, outbound {int(G11.out_degree(c, weight='weight'))}")
    else:
        print(f"{c}: not in network")

SEA-11 network: 73 nodes, 166 edges
(main analysis was 71 nodes, 158 edges)
MYS: inbound 169, outbound 0
SGP: not in network
BRN: not in network
TLS: not in network


In [4]:
bc_unwtd = nx.betweenness_centrality(G11)

for u, v, d in G11.edges(data=True):
    d["distance"] = 1 / d["weight"]
bc_wtd = nx.betweenness_centrality(G11, weight="distance")

rows = []
for c in sea11:
    if c in G11:
        rows.append({
            "country": c,
            "betw_unwtd": round(bc_unwtd[c], 4),
            "betw_wtd": round(bc_wtd[c], 4),
            "records": int(G11.in_degree(c, weight="weight") + G11.out_degree(c, weight="weight")),
        })

out = pd.DataFrame(rows).sort_values("betw_wtd", ascending=False)
print(out.to_string(index=False))
print("\nMain analysis weighted: KHM 0.1716 | THA 0.1653 | IDN 0.1290 | VNM 0.1179 | PHL 0.0963")

country  betw_unwtd  betw_wtd  records
    KHM      0.1117    0.1622      154
    THA      0.1033    0.1556      223
    IDN      0.0981    0.1213      127
    VNM      0.0854    0.1114      518
    PHL      0.1023    0.0908      298
    MMR      0.0441    0.0646      118
    LAO      0.0000    0.0000        8
    MYS      0.0000    0.0000      169

Main analysis weighted: KHM 0.1716 | THA 0.1653 | IDN 0.1290 | VNM 0.1179 | PHL 0.0963
